# AI Gateway ❤️ Foundry

## AI Gateway Foundry Models lab
![flow](../../images/aigw-foundry-models.gif)

Playground to try the [**AI Gateway tier of Azure API Management (public preview)**](https://techcommunity.microsoft.com/blog/integrationsonazureblog/ai-gateway-tier-of-api-management-now-in-public-preview/4540170) — a purpose-built experience for publishing and governing AI models and MCP servers. This lab focuses on publishing Microsoft Foundry models through a gateway-managed endpoint, where governance is configured through policy cards (token rate limits, quotas, Azure AI Content Safety, and model fallback) rather than XML. Applications call the published models under a single stable endpoint using standard formats such as OpenAI Chat Completions and Responses, while OpenTelemetry-based token-usage metrics flow to the Application Insights resource.

> The AI Gateway tier is in public preview and available in **East US 2** and **Sweden Central**.

> [!WARNING]
> **Public preview disclaimer:** The AI Gateway tier of Azure API Management is currently in **public preview**. Preview features are provided without a service-level agreement and are not recommended for production workloads. Certain features might not be supported or might have constrained capabilities, and behavior, APIs, and pricing may change before general availability. For details, review the [Supplemental Terms of Use for Microsoft Azure Previews](https://azure.microsoft.com/support/legal/preview-supplemental-terms/).

### Contents

1. [Initialize notebook variables](#0)
1. [Verify the Azure CLI and the connected Azure subscription](#1)
1. [Create deployment using Bicep](#2)
1. [Get the deployment outputs](#3)
1. [Grant the AI Gateway access to the managed Data Collection Rule](#dcr-role)
1. [Test with the OpenAI Python SDK (chat completions)](#openai-sdk)
1. [Test with the OpenAI Responses API](#responses-api)
1. [Test streaming responses (SSE) — time to first token](#streaming)
1. [Test authentication — reject missing / invalid keys](#auth)
1. [Demonstrate the token rate-limit policy](#rate-limit)
1. [Visualize the token rate limit](#rate-limit-graph)
1. [Get Prometheus endpoint and access token](#prometheus)
1. [Display token usage](#query)
1. [Token usage breakdown by model and subscription](#usage-breakdown)
1. [Clean up resources](#clean)

### Prerequisites

- [Python 3.12 or later version](https://www.python.org/) installed
- [VS Code](https://code.visualstudio.com/) installed with the [Jupyter notebook extension](https://marketplace.visualstudio.com/items?itemName=ms-toolsai.jupyter) enabled
- [uv](https://docs.astral.sh/uv/) — run `uv sync` from the repo root to install dependencies
- [An Azure Subscription](https://azure.microsoft.com/free/) with [Contributor](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#contributor) + [RBAC Administrator](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#role-based-access-control-administrator) or [Owner](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#owner) roles
- [Azure CLI](https://learn.microsoft.com/cli/azure/install-azure-cli) installed and [Signed into your Azure subscription](https://learn.microsoft.com/cli/azure/authenticate-azure-cli-interactively)

▶️ Click `Run All` to execute all steps sequentially, or execute them `Step by Step`... 


<a id='0'></a>
### 0️⃣ Initialize notebook variables

- Resources will be suffixed by a unique string based on your subscription id.
- Adjust the location parameters according your preferences and on the [product availability by Azure region.](https://azure.microsoft.com/explore/global-infrastructure/products-by-region/?cdn=disable&products=cognitive-services,api-management) 
- Adjust the models and versions according the [availability by region.](https://learn.microsoft.com/azure/ai-services/openai/concepts/models) 

In [ ]:
import os, sys, json
sys.path.insert(1, '../../shared')  # add the shared directory to the Python path
import utils

deployment_name = os.path.basename(os.path.dirname(globals()['__vsc_ipynb_file__']))
resource_group_name = f"lab-{deployment_name}" # change the name to match your naming style
resource_group_location = "swedencentral"

workspace_name = 'default'

foundry_accounts_config = [{"name": "foundry1", "location": "swedencentral"}]

foundry_project_name = deployment_name

models_config = [{"name": "model-router", "format": "OpenAI", "version": "2025-05-19", "sku": "GlobalStandard", "capacity": 20, "policies": [ {"type": "tokenLimit", "period": "minute", "count": 1000, "counterKey": "Identity"}]},
                 {"name": "gpt-5.6-sol", "format": "OpenAI", "version": "2026-07-09", "sku": "GlobalStandard", "capacity": 50, "policies": [ {"type": "tokenLimit", "period": "minute", "count": 1000, "counterKey": "Identity"}]},
                 {"name": "Kimi-K2.6", "format": "MoonshotAI", "version": "2026-04-20", "sku": "GlobalStandard", "capacity": 50, "policies": [ {"type": "tokenLimit", "period": "minute", "count": 1000, "counterKey": "Identity"}]}]

api_keys_config = [{"name": "key1", "displayName": "API Key 1"}, 
                   {"name": "key2", "displayName": "API Key 2"}, 
                   {"name": "key3", "displayName": "API Key 3"}]

payloadCapture = True  # Set to True to enable payload capture for telemetry exporters

utils.print_ok('Notebook initialized')


<a id='1'></a>
### 1️⃣ Verify the Azure CLI and the connected Azure subscription

The following commands ensure that you have the latest version of the Azure CLI and that the Azure CLI is connected to your Azure subscription.

In [ ]:
output = utils.run("az account show", "Retrieved az account", "Failed to get the current az account")

if output.success and output.json_data:
    current_user = output.json_data['user']['name']
    tenant_id = output.json_data['tenantId']
    subscription_id = output.json_data['id']

    utils.print_info(f"Current user: {current_user}")
    utils.print_info(f"Tenant ID: {tenant_id}")
    utils.print_info(f"Subscription ID: {subscription_id}")

<a id='2'></a>
### 2️⃣ Create deployment using 🦾 Bicep

This lab uses [Bicep](https://learn.microsoft.com/azure/azure-resource-manager/bicep/overview?tabs=bicep) to declarative define all the resources that will be deployed in the specified resource group. Change the parameters or the [main.bicep](main.bicep) directly to try different configurations. 

In [ ]:
# Create the resource group if doesn't exist
utils.create_resource_group(resource_group_name, resource_group_location)

# Define the Bicep parameters
bicep_parameters = {
    "$schema": "https://schema.management.azure.com/schemas/2019-04-01/deploymentParameters.json#",
    "contentVersion": "1.0.0.0",
    "parameters": {
        "workspaceName": { "value": workspace_name },
        "foundryAccountsConfig": { "value": foundry_accounts_config },
        "foundryProjectName": { "value": foundry_project_name },
        "modelsConfig": { "value": models_config },
        "apiKeysConfig": { "value": api_keys_config },
        "payloadCapture": { "value": payloadCapture }
    }
}

# Write the parameters to the params.json file
with open('params.json', 'w') as bicep_parameters_file:
    bicep_parameters_file.write(json.dumps(bicep_parameters))

# Run the deployment
output = utils.run(f"az deployment group create --name {deployment_name} --resource-group {resource_group_name} --template-file main.bicep --parameters params.json",
    f"Deployment '{deployment_name}' succeeded", f"Deployment '{deployment_name}' failed")

<a id='3'></a>
### 3️⃣ Get the deployment outputs

We are now at the stage where we only need to retrieve the gateway URL and the subscription before we are ready for testing.

In [ ]:
# Obtain all of the outputs from the deployment
output = utils.run(f"az deployment group show --name {deployment_name} -g {resource_group_name}", f"Retrieved deployment: {deployment_name}", f"Failed to retrieve deployment: {deployment_name}")

if output.success and output.json_data:
    log_analytics_id = utils.get_deployment_output(output, 'logAnalyticsWorkspaceId', 'Log Analytics Id')
    apim_service_id = utils.get_deployment_output(output, 'apimServiceId', 'APIM Service Id')
    apim_resource_gateway_url = utils.get_deployment_output(output, 'apimResourceGatewayURL', 'APIM API Gateway URL')
    gateway_principal_id = utils.get_deployment_output(output, 'gatewayPrincipalId', 'AI Gateway Principal Id')
    app_insights_id = utils.get_deployment_output(output, 'appInsightsId', 'Application Insights Id')
    apim_keys = json.loads(utils.get_deployment_output(output, 'apiKeys').replace("\'", "\""))
    for key in apim_keys:
        key_name = key['name']
        key_value = key['key']
        utils.print_info(f"Key Name: {key_name}")
        utils.print_info(f"Key Value: ****{key_value[-4:]}")
    api_key = apim_keys[0].get("key") # default api key to the first subscription key
    base_url = f"{apim_resource_gateway_url}/{workspace_name}/models/openai/v1"


<a id='dcr-role'></a>
### 🔐 Grant the AI Gateway access to the managed Data Collection Rule

The AI Gateway managed identity needs the **Monitoring Metrics Publisher** role on the managed Data Collection Rule (DCR) that Application Insights provisions. The DCR sits in an Azure‑managed resource group protected by a *deny assignment*, which blocks ARM/Bicep deployments into it — so the role is assigned here with a direct `az role assignment create` call (the same operation the portal performs).


In [ ]:
# Grant the AI Gateway managed identity the "Monitoring Metrics Publisher" role on the
# managed Data Collection Rule (DCR). The DCR lives in an Azure-managed resource group
# protected by a deny assignment that blocks ARM/Bicep deployments, so the role is
# assigned with a direct CLI call — the same operation the Azure portal performs.

# Resolve the managed DCR resource id from the Application Insights component
dcr_output = utils.run(f'az resource show --ids "{app_insights_id}" --query "properties.DataCollectionRuleResourceId" -o tsv')
data_collection_rule_id = dcr_output.text.strip()

if data_collection_rule_id:
    utils.print_info(f"Data Collection Rule Id: {data_collection_rule_id}")
    utils.run(
        f'az role assignment create --assignee-object-id {gateway_principal_id} --assignee-principal-type ServicePrincipal '
        f'--role "Monitoring Metrics Publisher" --scope "{data_collection_rule_id}"',
        "Assigned 'Monitoring Metrics Publisher' on the Data Collection Rule",
        "Failed to assign 'Monitoring Metrics Publisher' on the Data Collection Rule")
else:
    utils.print_info("No Data Collection Rule is associated with Application Insights yet.")


<a id='openai-sdk'></a>
### 🧪 Test with the OpenAI Python SDK (chat completions)

Call a published model through the gateway using the OpenAI **Chat Completions** format. We request a [structured output](https://platform.openai.com/docs/guides/structured-outputs) by passing the `TimeResponse` Pydantic schema and read the parsed result.


In [ ]:
from openai import OpenAI
from pydantic import BaseModel

# Define the structured output schema using a Pydantic model
class TimeResponse(BaseModel):
    current_time: str  # The time in HH:MM 24-hour format
    sarcastic_remark: str  # A sarcastic comment to accompany the answer

client = OpenAI(api_key=api_key, base_url=base_url, default_headers={"api-key": api_key})

# `.parse()` enforces the schema and returns a typed object.
completion = client.chat.completions.with_raw_response.parse(
    model=models_config[0]['name'],
    messages=[
        {"role": "system", "content": "You are a sarcastic, unhelpful assistant."},
        {"role": "user", "content": "Can you tell me the time, please?"}
    ],
    response_format=TimeResponse
)

# The x-ms-* headers reveal which backend/region the gateway routed the request to
print("x-ms-region:", completion.headers.get("x-ms-region"))

result = completion.parse().choices[0].message.parsed
print(f"💬 {result}")


<a id='responses-api'></a>
### 🧪 Test with the OpenAI Responses API

The AI Gateway also publishes the models under the [OpenAI **Responses API**](https://platform.openai.com/docs/api-reference/responses) format at the same stable endpoint. Below we call `client.responses.create` and read the aggregated `output_text`.


In [ ]:
from openai import OpenAI

client = OpenAI(api_key=api_key, base_url=base_url, default_headers={"api-key": api_key})

# Call the same model via the Responses API format.
response = client.responses.with_raw_response.create(
    model=models_config[1]['name'],
    input=[
        {"role": "system", "content": "You are a sarcastic, unhelpful assistant."},
        {"role": "user", "content": "Can you tell me the time, please?"}
    ]
)

print(f"💬 {response.parse().output_text}")


<a id='streaming'></a>
### 🧪 Test streaming responses (SSE) — time to first token

Stream a chat completion via the gateway and measure **time-to-first-token (TTFT)** and total time. Streaming returns incremental [server-sent events](https://developer.mozilla.org/docs/Web/API/Server-sent_events), so the first token arrives well before the full response — a key latency metric for interactive apps.


In [ ]:
import time
from openai import OpenAI

client = OpenAI(api_key=api_key, base_url=base_url, default_headers={"api-key": api_key})

start = time.time()
first_token_at = None
chunks, text = 0, ""

stream = client.chat.completions.create(
    model=models_config[1]['name'],
    messages=[{"role": "user", "content": "Count from 1 to 30, adding a short fact about each number."}],
    stream=True,
)

for chunk in stream:
    delta = chunk.choices[0].delta.content if chunk.choices else None
    if delta:
        if first_token_at is None:
            first_token_at = time.time() - start
            print(f"⚡ Time to first token: {first_token_at:.2f}s")
        chunks += 1
        text += delta

total = time.time() - start
print(f"📦 Chunks: {chunks} | ⌚ Total: {total:.2f}s | streamed {len(text)} chars")
print(f"💬 {text[:300]}{'...' if len(text) > 300 else ''}")


<a id='auth'></a>
### 🧪 Test authentication — reject missing / invalid keys

The gateway requires a valid subscription key. Verify that a **missing** or **invalid** `api-key` is rejected with **HTTP 401 Unauthorized**, while a valid key succeeds.


In [ ]:
import requests

url = f"{base_url}/chat/completions"
payload = {"model": models_config[1]['name'], "messages": [{"role": "user", "content": "hello"}]}

# (label, headers, expected status)
cases = [
    ("missing api-key", {"Content-Type": "application/json"}, 401),
    ("invalid api-key", {"api-key": "not-a-real-key", "Content-Type": "application/json"}, 401),
    ("valid api-key",   {"api-key": api_key, "Content-Type": "application/json"}, 200),
]

all_passed = True
for label, hdrs, expected in cases:
    resp = requests.post(url, headers=hdrs, json=payload)
    passed = resp.status_code == expected
    all_passed &= passed
    print(f"{'✅' if passed else '❌'} {label:>16}: HTTP {resp.status_code} (expected {expected})")

print("\n🎉 Authentication enforced correctly." if all_passed
      else "\n⚠️ Unexpected auth result — inspect above.")


<a id='rate-limit'></a>
### 🧪 Demonstrate the token rate-limit policy

Each model is published with a **token limit** policy card (`tokenLimit`, per identity). The gateway enforces it as a **leaky bucket**: consumed tokens fill a bucket of size `count` that drains continuously at `count / 60` tokens per second. When a request arrives and the bucket is already full, the gateway responds with **HTTP 429 Too Many Requests** plus a `Retry-After` header indicating how long until enough has drained.

Because completion tokens are only known *after* the model responds, the last accepted request can push the bucket slightly past the limit — so you'll typically see the usage overshoot `count` by roughly one request before the first 429.

The gateway does **not** return a remaining-tokens header, so the loop below reproduces the same leaky-bucket math client-side to estimate the remaining allowance.


In [ ]:
import time, requests

model = models_config[0]
url = f"{base_url}/chat/completions"
headers = {"api-key": api_key, "Content-Type": "application/json"}

# Read the per-minute token budget from the model's tokenLimit policy card
token_limit = next((p["count"] for p in model.get("policies", []) if p["type"] == "tokenLimit"), 1000)

# Leaky-bucket model: the bucket holds consumed tokens and drains continuously.
leak_rate = token_limit / 60.0   # tokens that "leak" (free up) every second
bucket = 0.0                     # current fill level of the bucket
last_update = time.time()        # last time we drained the bucket

run_duration = 60       # run the test for one minute
rate_limit_runs = []    # collect per-run results so the next cell can plot them
loop_start = time.time()   # reference point to measure elapsed seconds on the leaky bucket

i = 0
while time.time() - loop_start < run_duration:
    i += 1
    response = requests.post(url, headers=headers, json={
        "model": model['name'],
        "messages": [{"role": "user", "content": "Write a 200 token essay."}]
    })

    now = time.time()
    elapsed = now - loop_start                       # seconds since the loop started
    bucket = max(0.0, bucket - leak_rate * (now - last_update))   # drain since last update
    last_update = now

    if response.status_code == 429:
        # The AI Gateway returns HTTP 429 once the bucket is full (per-minute token budget exhausted).
        # Honor the Retry-After header: wait for that many seconds, then keep going.
        retry_after = int(response.headers.get("Retry-After", 1))
        rate_limit_runs.append({"run": i, "status": 429, "remaining_tokens": 0,
                                "bucket": round(bucket), "elapsed": round(elapsed, 1)})
        print(f"⏱️ {elapsed:6.1f}s | 🚦 Rate limit hit on run {i} — HTTP 429 Too Many Requests. "
              f"Retry-After: {retry_after}s → sleeping...")
        time.sleep(retry_after)
        continue

    body = response.json()   # read the model and token usage from the response body
    usage = body.get("usage", {})

    # Add the tokens this request consumed to the bucket; remaining is the free capacity.
    bucket += usage.get("total_tokens", 0)
    remaining_tokens = max(0, round(token_limit - bucket))

    rate_limit_runs.append({"run": i, "status": response.status_code, "remaining_tokens": remaining_tokens,
                            "bucket": round(bucket), "elapsed": round(elapsed, 1)})
    print(f"⏱️ {elapsed:6.1f}s | ▶️ Run {i:>2}: HTTP {response.status_code} | model: {body.get('model')} | "
          f"prompt tokens: {usage.get('prompt_tokens')} | completion tokens: {usage.get('completion_tokens')} | "
          f"bucket: {round(bucket)}/{token_limit} | remaining tokens: {remaining_tokens}")

# --- Validate that the leaky-bucket rate limit behaved as expected -------------------
ok_runs = [r for r in rate_limit_runs if r["status"] == 200]
throttled = [r for r in rate_limit_runs if r["status"] == 429]
peak_bucket = max((r["bucket"] for r in ok_runs), default=0)
# a successful call after a throttle proves the bucket drained and traffic resumed
recovered = any(r["status"] == 200 and r["elapsed"] > throttled[0]["elapsed"] for r in rate_limit_runs) if throttled else False

checks = [
    (len(ok_runs) > 0,                 f"requests succeed while under budget ({len(ok_runs)} × HTTP 200)"),
    (len(throttled) > 0,               f"policy throttles once the bucket fills ({len(throttled)} × HTTP 429)"),
    (peak_bucket >= token_limit,       f"throttling starts at the limit (peak bucket {peak_bucket} ≥ {token_limit})"),
    (recovered,                        "traffic resumes after the bucket drains (HTTP 200 following a 429)"),
]

print("\n" + "─" * 60)
print("Leaky-bucket rate-limit validation:")
for passed, description in checks:
    print(f"  {'✅' if passed else '❌'} {description}")
print("🎉 Working as expected — the token rate limit is enforced correctly."
      if all(p for p, _ in checks) else
      "⚠️ Unexpected result — inspect the run log above.")


<a id='rate-limit-graph'></a>
### 📊 Visualize the token rate limit

Plot the estimated remaining token allowance (leaky-bucket free capacity) on each run. It drops as requests fill the bucket and recovers as the bucket drains during the `Retry-After` waits — producing a sawtooth pattern. Runs that received **HTTP 429** are highlighted at zero remaining.


In [ ]:
import matplotlib.pyplot as plt

if not rate_limit_runs:
    utils.print_error("No data to plot. Run the rate-limit cell above first.")
else:
    t = [r["elapsed"] for r in rate_limit_runs]
    bucket_level = [r["bucket"] for r in rate_limit_runs]

    ok = [r for r in rate_limit_runs if r["status"] == 200]
    throttled = [r for r in rate_limit_runs if r["status"] == 429]

    fig, ax = plt.subplots(figsize=(11, 5))

    # Bucket fill level over time — rises with each request, drains during Retry-After waits
    ax.plot(t, bucket_level, marker="o", color="#0078D4", label="Bucket fill (tokens)")

    # The limit line: at/above this the gateway throttles
    ax.axhline(token_limit, color="#8A8886", linestyle="--", label=f"Token limit ({token_limit})")

    # Highlight accepted vs throttled requests
    ax.scatter([r["elapsed"] for r in ok], [r["bucket"] for r in ok],
               color="#107C10", zorder=3, label="HTTP 200 (accepted)")
    ax.scatter([r["elapsed"] for r in throttled], [r["bucket"] for r in throttled],
               color="#D13438", s=120, zorder=4, label="HTTP 429 (throttled)")

    ax.set_title("AI Gateway token rate limit — leaky bucket over time")
    ax.set_xlabel("Elapsed time (s)")
    ax.set_ylabel("Bucket fill (tokens)")
    ax.set_ylim(bottom=0)
    ax.grid(True, alpha=0.3)
    ax.legend(loc="upper right")
    plt.tight_layout()
    plt.show()

    # Confirm visually + numerically that throttling aligns with the limit
    if throttled:
        print(f"✅ Every HTTP 429 occurred at or above the {token_limit}-token limit "
              f"(bucket ≥ {min(r['bucket'] for r in throttled)}), and the bucket drained "
              f"back down between throttles — leaky-bucket behavior confirmed.")


<a id='prometheus'></a>
### ⚙️ Get Prometheus endpoint and access token


In [ ]:
data_collection_rule_id = utils.run(
    f'az resource show --ids "{app_insights_id}" --query "properties.DataCollectionRuleResourceId" -o tsv').text.strip()
monitoring_account_id = utils.run(
    f'az resource show --ids "{data_collection_rule_id}" --query "properties.destinations.monitoringAccounts[0].accountResourceId" -o tsv').text.strip()
prometheus_query_endpoint = utils.run(
    f'az resource show --ids "{monitoring_account_id}" --query "properties.metrics.prometheusQueryEndpoint" -o tsv').text.strip()
utils.print_info(f"Prometheus query endpoint: {prometheus_query_endpoint}")

prometheus_access_token = utils.run(
    'az account get-access-token --resource https://prometheus.monitor.azure.com --query accessToken -o tsv').text.strip()


<a id='query'></a>
### 🔍 Display token usage

The AI Gateway emits OpenTelemetry metrics that are ingested into an Azure Monitor Workspace (managed Prometheus). We query the token-usage metric via the Prometheus API.

> ⏱️ Metrics can take a few minutes to appear after the first request. If the table below is empty, wait a moment and re-run the cell.


In [ ]:
import pandas as pd, requests, time

metric = "azure.ai_gateway.client.token.usage"
selector = (f'{{__name__="{metric}","microsoft.appresourceid"="{app_insights_id.lower()}",'
            f'"azure.ai_gateway.workspace"="{workspace_name}"}}')

# Instant query at 'now' over a 2h window — same as the portal
r = requests.get(f"{prometheus_query_endpoint}/api/v1/query",
                 headers={"Authorization": f"Bearer {prometheus_access_token}"},
                 params={"query": f"increase({selector}[7200s])", "time": int(time.time())})

series = r.json().get("data", {}).get("result", []) if r.status_code == 200 else None

if series:
    df = pd.DataFrame([{**s["metric"], "TokenUsage": float(s["value"][1])} for s in series])
    # keep only the prompt / completion / total token types
    token_types = {"prompt_tokens", "completion_tokens", "total_tokens"}
    type_col = next((c for c in df.columns if token_types & set(df[c].dropna())), None)
    if type_col:
        df = df[df[type_col].isin(token_types)]
    display(df)
elif r.status_code != 200:
    utils.print_error(f"Prometheus query failed ({r.status_code})", r.text)
else:
    utils.print_error("No data — metrics may still be propagating (allow a few minutes), or check the label values against the portal query.")


<a id='usage-breakdown'></a>
### 🔍 Token usage breakdown by model and subscription

Aggregate the same token-usage metric grouped by the model and subscription/identity labels, so you can see which model and which caller consumed the tokens — not just the overall total.


In [ ]:
import pandas as pd, requests, time

metric = "azure.ai_gateway.client.token.usage"
selector = (f'{{__name__="{metric}","microsoft.appresourceid"="{app_insights_id.lower()}",'
            f'"azure.ai_gateway.workspace"="{workspace_name}"}}')

r = requests.get(f"{prometheus_query_endpoint}/api/v1/query",
                 headers={"Authorization": f"Bearer {prometheus_access_token}"},
                 params={"query": f"increase({selector}[7200s])", "time": int(time.time())})

series = r.json().get("data", {}).get("result", []) if r.status_code == 200 else []

if series:
    df = pd.DataFrame([{**s["metric"], "TokenUsage": float(s["value"][1])} for s in series])
    # detect the label columns that identify the model and the caller (subscription/identity/key)
    dims = [c for c in df.columns if any(k in c.lower() for k in ("model", "subscription", "identity", "key"))]
    if dims:
        summary = (df.groupby(dims, dropna=False)["TokenUsage"].sum()
                     .reset_index().sort_values("TokenUsage", ascending=False))
        display(summary)
    else:
        print("No model/subscription labels present. Available labels:",
              [c for c in df.columns if c != "TokenUsage"])
        display(df)
else:
    utils.print_error("No token-usage data returned (metrics may still be propagating).")


<a id='clean'></a>
### 🗑️ Clean up resources

When you're finished with the lab, you should remove all your deployed resources from Azure to avoid extra charges and keep your Azure subscription uncluttered.
Use the [clean-up-resources notebook](clean-up-resources.ipynb) for that.